# 🧬 Tsukumo — Monte Carlo What-If Scenario Engine

> **"This isn't a trained model in the traditional sense — it's a statistical simulation layer"**
>
> This notebook implements a **Synthetic Data Generation (SDG) + Monte Carlo Simulation** pipeline. It:
> 1. Synthesises thousands of *virtual versions* of a patient using **Framingham & NHANES** statistical priors
> 2. Injects stochastic lifestyle shocks ("What if I run 3× a week?", "What if I cut sodium?")
> 3. Runs each virtual cohort through the **cardiac-arrest.pkl** and **diabetes.pkl** models
> 4. Produces **probability distributions** with 80/95% confidence intervals and projected 6-month trajectories

---
**References**
- Framingham Heart Study: D'Agostino et al. (2008), NHANES 2017–2020
- Simulation method: [Monte Carlo Methods in Risk Analysis](https://doi.org/10.1007/s00477-013-0695-3)


In [ ]:
# ── 0. Install / check dependencies ──────────────────────────────────────────
import subprocess, sys

REQUIRED = [
    "numpy", "pandas", "scipy", "scikit-learn",
    "matplotlib", "seaborn", "joblib", "tqdm", "ipywidgets"
]

for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"  Installing {pkg}…")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅  All dependencies satisfied.")

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm
from dataclasses import dataclass, field
from typing import Dict, Optional, List

warnings.filterwarnings("ignore")

# ── Plotting aesthetics ───────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f0f1a",
    "axes.facecolor":   "#16162a",
    "axes.edgecolor":   "#2a2a4a",
    "axes.labelcolor":  "#c8c8e8",
    "axes.titlecolor":  "#e8e8ff",
    "axes.titlesize":   14,
    "axes.labelsize":   11,
    "xtick.color":      "#8888aa",
    "ytick.color":      "#8888aa",
    "text.color":       "#c8c8e8",
    "grid.color":       "#2a2a4a",
    "grid.linestyle":   "--",
    "legend.facecolor": "#1e1e38",
    "legend.edgecolor": "#3a3a6a",
    "font.family":      "DejaVu Sans",
})

PALETTE = {
    "baseline": "#7b7bff",
    "best":     "#2ecc71",
    "worst":    "#e74c3c",
    "moderate": "#f39c12",
    "fill_lo":  "#2ecc711a",
    "fill_hi":  "#e74c3c1a",
}

print("✅  Imports complete.")

---
## 🧱 Section 1 — Framingham / NHANES Statistical Priors

These distributions encode the **population-level** statistics for adult Americans (NHANES 2017-2020) and the Framingham Heart Study cohort.  
Every synthetic patient will be drawn from these priors and then *conditioned* on the real user's baseline measurements.

In [ ]:
# ── 2. Population Priors  ─────────────────────────────────────────────────────
# Format: { feature: (mean, std, low_clip, high_clip) }
#
# Sources:
#   NHANES 2017-2020: https://wwwn.cdc.gov/nchs/nhanes/
#   Framingham Heart Study: D'Agostino et al. (2008) Circ 117:743-753

NHANES_PRIORS: Dict[str, tuple] = {
    # ── Demographics ──────────────────────────────────────────────────────────
    "age":                       (47.8,  18.0,  18,  90),
    "bmi":                       (29.6,   7.0,  15,  60),

    # ── Cardiovascular ────────────────────────────────────────────────────────
    "sysBP":                     (124.0, 19.0,  80, 220),   # Systolic BP (mmHg)
    "diaBP":                     ( 76.0, 12.0,  50, 130),   # Diastolic BP (mmHg)
    "heartRate":                 ( 73.0, 13.0,  40, 140),   # BPM
    "totChol":                   (197.0, 40.0, 100, 400),   # mg/dL
    "HDLChol":                   ( 54.0, 16.0,  20, 110),   # mg/dL
    "glucose":                   (104.0, 30.0,  60, 400),   # mg/dL fasting

    # ── Lifestyle ─────────────────────────────────────────────────────────────
    "cigarettes_per_day":        (  3.5,  7.0,   0,  60),   # avg incl. non-smokers
    "sleep_hours":               (  6.9,  1.4,   3,  12),
    "physical_activity_met":     (  2.8,  1.6,   1,  12),   # MET-hrs/week
    "sodium_mg_per_day":         (3440,  900,  500, 8000),
    "alcohol_drinks_per_week":   (  3.1,  5.5,   0,  50),

    # ── Metabolic ─────────────────────────────────────────────────────────────
    "prevalentHyp":              (  0.45,  0.497, 0, 1),    # 0/1 binary
    "diabetes":                  (  0.11,  0.31,  0, 1),
    "currentSmoker":             (  0.14,  0.35,  0, 1),
}

# Binary feature names (clamped to 0/1 after sampling)
BINARY_FEATURES = {"prevalentHyp", "diabetes", "currentSmoker"}

print(f"✅  {len(NHANES_PRIORS)} population priors loaded.")

---
## 🏥 Section 2 — Model Loader

We load the same **cardiac-arrest.pkl** and **diabetes.pkl** models used by the Flask micro-service.  
If either model is absent, a **logistic-regression fallback** trained on synthetic data is used so the notebook always runs end-to-end.

In [ ]:
# ── 3. Model Loader ───────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
PUBLIC_DIR   = os.path.join(PROJECT_ROOT, "public")

def load_pkl(path: str):
    """Load a pickle model; return None if absent."""
    if os.path.exists(path):
        with open(path, "rb") as f:
            model = pickle.load(f)
        print(f"  ✅  Loaded: {os.path.basename(path)}")
        return model
    else:
        print(f"  ⚠️  Not found: {path}  → using synthetic fallback")
        return None

cardiac_model  = load_pkl(os.path.join(PUBLIC_DIR, "cardiac-arrest.pkl"))
diabetes_model = load_pkl(os.path.join(PUBLIC_DIR, "diabetes.pkl"))


# ── Synthetic fallback model ──────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def _build_fallback_model(feature_names: list, seed: int = 42) -> Pipeline:
    """Trains a logistic-regression on randomly drawn synthetic data."""
    rng = np.random.default_rng(seed)
    n   = 5_000
    X   = rng.standard_normal((n, len(feature_names)))
    # Risk increases with sysBP, totChol, age, bmi, glucose
    logit = (
        0.03 * X[:, feature_names.index("sysBP")]    +
        0.02 * X[:, feature_names.index("totChol")]  +
        0.04 * X[:, feature_names.index("age")]      +
        0.02 * X[:, feature_names.index("bmi")]      +
        0.02 * X[:, feature_names.index("glucose")]  - 1.5
    )
    y   = (1 / (1 + np.exp(-logit)) > 0.5).astype(int)
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("clf",    LogisticRegression(max_iter=500))])
    pipe.fit(X, y)
    return pipe

# Feature order used when calling models
CARDIAC_FEATURES = [
    "age", "sysBP", "diaBP", "heartRate", "totChol", "HDLChol",
    "glucose", "bmi", "currentSmoker", "cigarettes_per_day", "prevalentHyp"
]
DIABETES_FEATURES = [
    "age", "bmi", "glucose", "sysBP", "diaBP", "totChol",
    "HDLChol", "sleep_hours", "physical_activity_met", "sodium_mg_per_day"
]

if cardiac_model is None:
    cardiac_model  = _build_fallback_model(CARDIAC_FEATURES,  seed=1)
    print("  🛠️  Using synthetic logistic-regression fallback for cardiac model")
if diabetes_model is None:
    diabetes_model = _build_fallback_model(DIABETES_FEATURES, seed=2)
    print("  🛠️  Using synthetic logistic-regression fallback for diabetes model")

print()
print("✅  Both models ready.")

---
## 🧪 Section 3 — Synthetic Data Generation (SDG)

We generate a virtual **cohort** of `N_SIMULATIONS` patients.  
Each patient is drawn from the NHANES/Framingham multivariate normal prior, then **conditioned** so the population mean equals the real user's measured values. This is equivalent to a **Bayesian update** of the population prior given observed data.

In [ ]:
# ── 4. SDG Engine ─────────────────────────────────────────────────────────────

@dataclass
class PatientProfile:
    """Real observed measurements for the user — edit to match your patient."""
    # ── Demographics ──────────────────────────────────────────────────────────
    age:                     float = 45.0
    bmi:                     float = 28.5

    # ── Cardiovascular ────────────────────────────────────────────────────────
    sysBP:                   float = 140.0
    diaBP:                   float = 90.0
    heartRate:               float = 85.0
    totChol:                 float = 215.0
    HDLChol:                 float = 45.0
    glucose:                 float = 160.0

    # ── Lifestyle ─────────────────────────────────────────────────────────────
    cigarettes_per_day:      float = 0.0
    sleep_hours:             float = 6.0
    physical_activity_met:   float = 1.5
    sodium_mg_per_day:       float = 3800.0
    alcohol_drinks_per_week: float = 4.0

    # ── Metabolic flags ───────────────────────────────────────────────────────
    prevalentHyp:            float = 1.0
    diabetes:                float = 0.0
    currentSmoker:           float = 0.0


def generate_cohort(
    profile: PatientProfile,
    n: int = 10_000,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Draw n synthetic patients from NHANES priors, then shift the distribution
    so it is centred on the real patient's observed measurements.

    This implements a "Bayesian personalisation" step: the population spread
    (variance) is preserved but the mean is pulled to the individual's values.
    """
    rng = np.random.default_rng(seed)
    rows = {}

    for feat, (pop_mean, pop_std, lo, hi) in NHANES_PRIORS.items():
        # Draw from population
        samples = rng.normal(loc=pop_mean, scale=pop_std, size=n)

        # Shift centre to user's value (conditional mean update)
        user_val = getattr(profile, feat, pop_mean)
        delta    = user_val - pop_mean
        samples  = samples + delta

        # Clip to physiologically plausible range
        samples  = np.clip(samples, lo, hi)

        # Binarise if needed
        if feat in BINARY_FEATURES:
            samples = (samples > 0.5).astype(float)

        rows[feat] = samples

    return pd.DataFrame(rows)


# ── Demo: generate baseline cohort ───────────────────────────────────────────
N_SIMULATIONS = 10_000
PATIENT        = PatientProfile()     # ← Change this to your real measurements

baseline_cohort = generate_cohort(PATIENT, n=N_SIMULATIONS)
print(f"✅  Baseline cohort shape: {baseline_cohort.shape}")
print()
baseline_cohort[CARDIAC_FEATURES].describe().round(2)

---
## 💉 Section 4 — What-If Scenario Definitions

Each **scenario** is a named dictionary of *delta functions* that modify a copy of the cohort.  
Deltas can be absolute offsets, percentage changes, or hard overrides.

In [ ]:
# ── 5. Scenario Definitions ───────────────────────────────────────────────────

@dataclass
class Scenario:
    name:        str
    description: str
    color:       str
    # Each key is a feature name, value is a callable(array) -> array
    deltas:      Dict[str, object] = field(default_factory=dict)


def apply_scenario(cohort: pd.DataFrame, scenario: Scenario) -> pd.DataFrame:
    """Return a modified copy of the cohort with scenario deltas applied."""
    df = cohort.copy()
    for feat, fn in scenario.deltas.items():
        if feat in df.columns:
            lo, hi = NHANES_PRIORS[feat][2], NHANES_PRIORS[feat][3]
            df[feat] = np.clip(fn(df[feat].values), lo, hi)
    return df


# ─────────────────────────────────────────────────────────────────────────────
#  Define your what-if scenarios below
# ─────────────────────────────────────────────────────────────────────────────
SCENARIOS: List[Scenario] = [

    Scenario(
        name        = "Current Trajectory",
        description = "No changes — baseline measured values.",
        color       = PALETTE["baseline"],
        deltas      = {},    # <-- no modifications
    ),

    Scenario(
        name        = "Run 3× / Week",
        description = "Adds ≈9 MET-hrs/week of aerobic exercise. "
                      "Reduces systolic BP by ~5 mmHg, BMI by ~0.8, "
                      "heart rate by ~4 bpm (Framingham effect sizes).",
        color       = PALETTE["best"],
        deltas      = {
            "physical_activity_met": lambda x: x + 9.0,
            "sysBP":                 lambda x: x - 5.0,
            "bmi":                   lambda x: x - 0.8,
            "heartRate":             lambda x: x - 4.0,
            "glucose":               lambda x: x - 8.0,
            "totChol":               lambda x: x - 6.0,
            "HDLChol":               lambda x: x + 4.0,
        },
    ),

    Scenario(
        name        = "Cut Sodium to 2,300 mg/day",
        description = "Reduces sodium intake to recommended DASH-diet target. "
                      "Reduces systolic BP by ~4-6 mmHg.",
        color       = "#3498db",
        deltas      = {
            "sodium_mg_per_day": lambda x: np.minimum(x, 2300.0),
            "sysBP":             lambda x: x - 5.5,
            "diaBP":             lambda x: x - 2.5,
        },
    ),

    Scenario(
        name        = "Sleep 8 hrs / Night",
        description = "Achieves recommended 8 hrs. Linked to -3 mmHg SBP, "
                      "-5 mg/dL glucose (NHANES observational).",
        color       = "#9b59b6",
        deltas      = {
            "sleep_hours": lambda x: np.maximum(x, 8.0),
            "sysBP":       lambda x: x - 3.0,
            "glucose":     lambda x: x - 5.0,
        },
    ),

    Scenario(
        name        = "Full Lifestyle Optimisation",
        description = "Combines exercise + reduced sodium + optimal sleep.",
        color       = "#2ecc71",
        deltas      = {
            "physical_activity_met": lambda x: x + 9.0,
            "sodium_mg_per_day":     lambda x: np.minimum(x, 2300.0),
            "sleep_hours":           lambda x: np.maximum(x, 8.0),
            "sysBP":                 lambda x: x - 13.0,
            "diaBP":                 lambda x: x - 5.0,
            "bmi":                   lambda x: x - 0.8,
            "heartRate":             lambda x: x - 4.0,
            "glucose":               lambda x: x - 13.0,
            "totChol":               lambda x: x - 6.0,
            "HDLChol":               lambda x: x + 4.0,
        },
    ),

    Scenario(
        name        = "No Sodium Reduction",
        description = "Sodium increases by +500 mg/day (⚠️ worst-case dietary drift).",
        color       = PALETTE["worst"],
        deltas      = {
            "sodium_mg_per_day": lambda x: x + 500.0,
            "sysBP":             lambda x: x + 4.0,
            "diaBP":             lambda x: x + 2.0,
        },
    ),

    Scenario(
        name        = "Resume Smoking (10 cigs/day)",
        description = "Simulates relapse to light smoking. "
                      "Raises cardiac risk substantially.",
        color       = "#c0392b",
        deltas      = {
            "currentSmoker":    lambda x: np.ones_like(x),
            "cigarettes_per_day": lambda x: np.maximum(x, 10.0),
            "sysBP":            lambda x: x + 7.0,
            "totChol":          lambda x: x + 12.0,
            "HDLChol":          lambda x: x - 5.0,
        },
    ),
]

print(f"✅  {len(SCENARIOS)} scenarios defined:")
for s in SCENARIOS:
    print(f"   • {s.name}")

---
## 🎲 Section 5 — Monte Carlo Runner

For each scenario we run **10,000 virtual patients** through both predictive models, collecting the probability of a positive prediction across the cohort.  
The spread of that distribution gives us **confidence intervals** without requiring explicit uncertainty quantification of the model itself.

In [ ]:
# ── 6. Monte Carlo Runner ─────────────────────────────────────────────────────

def _safe_predict_proba(model, X: np.ndarray) -> np.ndarray:
    """Return probability of positive class, handling both sklearn and arbitrary models."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "predict"):
        preds = model.predict(X)
        return preds.astype(float)
    else:
        raise RuntimeError("Model has neither predict_proba nor predict.")


def run_monte_carlo(
    scenarios:      List[Scenario],
    baseline:       pd.DataFrame,
    cardiac_model,
    diabetes_model,
    cardiac_feats:  List[str],
    diabetes_feats: List[str],
) -> pd.DataFrame:
    """
    For each scenario:
      1. Apply deltas to the cohort
      2. Run cardiac + diabetes models
      3. Collect mean risk, 5th, 25th, 75th, 95th percentiles
    """
    records = []

    for sc in tqdm(scenarios, desc="Running scenarios"):
        cohort = apply_scenario(baseline, sc)

        # ── Cardiac ───────────────────────────────────────────────────────────
        try:
            X_c        = cohort[cardiac_feats].values.astype(float)
            c_probs    = _safe_predict_proba(cardiac_model, X_c)
        except Exception as e:
            print(f"  ⚠️  Cardiac model error for '{sc.name}': {e}")
            c_probs = np.zeros(len(cohort))

        # ── Diabetes ──────────────────────────────────────────────────────────
        try:
            X_d        = cohort[diabetes_feats].values.astype(float)
            d_probs    = _safe_predict_proba(diabetes_model, X_d)
        except Exception as e:
            print(f"  ⚠️  Diabetes model error for '{sc.name}': {e}")
            d_probs = np.zeros(len(cohort))

        for label, probs in [("cardiac", c_probs), ("diabetes", d_probs)]:
            records.append({
                "scenario":      sc.name,
                "risk_type":     label,
                "color":         sc.color,
                "mean_risk":     float(np.mean(probs)),
                "p05":           float(np.percentile(probs, 5)),
                "p25":           float(np.percentile(probs, 25)),
                "p50":           float(np.percentile(probs, 50)),
                "p75":           float(np.percentile(probs, 75)),
                "p95":           float(np.percentile(probs, 95)),
                "prob_drop_5pct": float(np.mean(probs < (np.mean(_safe_predict_proba(
                                    cardiac_model if label=="cardiac" else diabetes_model,
                                    baseline[cardiac_feats if label=="cardiac" else diabetes_feats].values.astype(float)
                                )) - 0.05))),
                "raw_probs":     probs,  # keep for distribution plots
            })

    return pd.DataFrame(records)


results = run_monte_carlo(
    SCENARIOS, baseline_cohort,
    cardiac_model, diabetes_model,
    CARDIAC_FEATURES, DIABETES_FEATURES,
)

print("\n✅  Monte Carlo complete. Summary:")
display_cols = ["scenario", "risk_type", "mean_risk", "p05", "p50", "p95"]
print(results[display_cols].to_string(index=False))

---
## 📊 Section 6 — Visualisations

In [ ]:
# ── 7. Plot A: Risk Probability Distribution per Scenario ─────────────────────

def plot_risk_distributions(results: pd.DataFrame, risk_type: str):
    df   = results[results["risk_type"] == risk_type].reset_index(drop=True)
    n_sc = len(df)

    fig, axes = plt.subplots(
        1, n_sc, figsize=(4.2 * n_sc, 4.5), sharey=False
    )
    fig.suptitle(
        f"{'🫀 Cardiac' if risk_type == 'cardiac' else '🩸 Diabetes'} Risk Distribution"
        " — Monte Carlo Scenarios",
        fontsize=15, fontweight="bold", y=1.02
    )

    for ax, (_, row) in zip(axes, df.iterrows()):
        probs = row["raw_probs"]
        color = row["color"]

        # KDE
        x = np.linspace(0, 1, 500)
        kde = stats.gaussian_kde(probs, bw_method=0.15)
        y   = kde(x)

        ax.fill_between(x, y, alpha=0.3, color=color)
        ax.plot(x, y, color=color, linewidth=2)

        # Percentile shading
        mask_80 = (x >= row["p05"]) & (x <= row["p95"])
        ax.fill_between(x[mask_80], y[mask_80], alpha=0.15, color=color)

        # Mean line
        ax.axvline(row["mean_risk"], color=color, linestyle="--", linewidth=1.5)
        ax.text(
            row["mean_risk"], ax.get_ylim()[1] * 0.6,
            f"μ={row['mean_risk']:.2%}",
            ha="center", fontsize=8, color=color,
        )

        ax.set_title(row["scenario"], fontsize=9, fontweight="bold")
        ax.set_xlabel("Predicted Risk Probability", fontsize=8)
        ax.set_ylabel("Density", fontsize=8)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.grid(True, alpha=0.2)

    fig.tight_layout()
    plt.show()


plot_risk_distributions(results, "cardiac")
plot_risk_distributions(results, "diabetes")

In [ ]:
# ── 8. Plot B: Scenario Comparison Bar Chart (80% CI) ─────────────────────────

def plot_scenario_comparison(results: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(
        "What-If Scenario Comparison — Mean Risk with 90% Confidence Interval",
        fontsize=14, fontweight="bold"
    )

    for ax, risk_type, title in zip(
        axes,
        ["cardiac", "diabetes"],
        ["🫀 Cardiac Arrest Risk", "🩸 Diabetes Risk"]
    ):
        df    = results[results["risk_type"] == risk_type].reset_index(drop=True)
        y_pos = np.arange(len(df))
        colors = df["color"].tolist()

        # Bars
        bars = ax.barh(
            y_pos,
            df["mean_risk"],
            color=colors,
            alpha=0.8,
            edgecolor="white",
            linewidth=0.5,
            height=0.55,
        )

        # CI error bars
        err_lo = df["mean_risk"] - df["p05"]
        err_hi = df["p95"]       - df["mean_risk"]
        ax.errorbar(
            df["mean_risk"], y_pos,
            xerr=[err_lo, err_hi],
            fmt="none", color="white", alpha=0.5,
            capsize=4, linewidth=1.2,
        )

        # Value labels
        for i, (_, row) in enumerate(df.iterrows()):
            ax.text(
                row["mean_risk"] + 0.003, i,
                f"{row['mean_risk']:.1%}",
                va="center", ha="left", fontsize=9, color="white"
            )

        ax.set_yticks(y_pos)
        ax.set_yticklabels(df["scenario"], fontsize=9)
        ax.set_xlabel("Predicted Risk Probability")
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.grid(True, axis="x", alpha=0.2)
        ax.invert_yaxis()

    fig.tight_layout()
    plt.show()


plot_scenario_comparison(results)

In [ ]:
# ── 9. Plot C: 6-Month Trajectory Projection ──────────────────────────────────
#
# We model the *gradual* effect of a lifestyle change over 24 weeks.
# Effect size ramps linearly from 0% (week 0) to 100% (week 24).
# At each week we re-run the MC simulation with a partially applied delta.

def project_trajectory(
    scenario:       Scenario,
    baseline:       pd.DataFrame,
    model,
    features:       List[str],
    n_weeks:        int   = 24,
    n_boot:         int   = 2_000,   # smaller cohort for speed
    seed:           int   = 99,
) -> pd.DataFrame:
    """
    Simulate how risk evolves week-by-week as the lifestyle change kicks in.
    """
    rng     = np.random.default_rng(seed)
    idx     = rng.choice(len(baseline), size=n_boot, replace=False)
    sub     = baseline.iloc[idx].copy()

    rows = []
    for week in range(n_weeks + 1):
        frac   = week / n_weeks          # 0.0 → 1.0
        cohort = sub.copy()

        for feat, fn in scenario.deltas.items():
            if feat in cohort.columns:
                original  = sub[feat].values
                target    = np.clip(
                    fn(original),
                    NHANES_PRIORS[feat][2],
                    NHANES_PRIORS[feat][3],
                )
                cohort[feat] = original + frac * (target - original)

        X     = cohort[features].values.astype(float)
        probs = _safe_predict_proba(model, X)
        rows.append({
            "week":    week,
            "mean":    np.mean(probs),
            "p10":     np.percentile(probs, 10),
            "p25":     np.percentile(probs, 25),
            "p75":     np.percentile(probs, 75),
            "p90":     np.percentile(probs, 90),
        })

    return pd.DataFrame(rows)


# Project 3 key scenarios for cardiac risk
TRAJECTORY_SCENARIOS = [SCENARIOS[0], SCENARIOS[1], SCENARIOS[4], SCENARIOS[5]]

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_title("🫀 6-Month Cardiac Risk Trajectory — Lifestyle Change Ramp-Up",
             fontsize=13, fontweight="bold")

for sc in tqdm(TRAJECTORY_SCENARIOS, desc="Projecting trajectories"):
    traj = project_trajectory(sc, baseline_cohort, cardiac_model, CARDIAC_FEATURES)
    weeks = traj["week"]

    ax.fill_between(weeks, traj["p10"], traj["p90"], alpha=0.10, color=sc.color)
    ax.fill_between(weeks, traj["p25"], traj["p75"], alpha=0.18, color=sc.color)
    ax.plot(weeks, traj["mean"], color=sc.color, linewidth=2.3, label=sc.name)

ax.set_xlabel("Week", fontsize=11)
ax.set_ylabel("Mean Predicted Cardiac Risk", fontsize=11)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, alpha=0.2)
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 4))
ax.set_xticklabels([f"Wk {w}" for w in range(0, 25, 4)])

fig.tight_layout()
plt.show()

In [ ]:
# ── 10. Plot D: Probability Statement Dashboard ────────────────────────────────
#
# Generates the key clinical probability statements:
#   "There is an X% chance your risk drops by ≥5% if you maintain [scenario]"

def compute_prob_statements(
    results:     pd.DataFrame,
    risk_type:   str,
    threshold:   float = 0.05,
) -> pd.DataFrame:
    """P(risk_scenario < risk_baseline - threshold)"""
    base_row   = results[
        (results["scenario"] == "Current Trajectory") &
        (results["risk_type"] == risk_type)
    ].iloc[0]
    base_probs = base_row["raw_probs"]
    base_mean  = np.mean(base_probs)

    rows = []
    for _, row in results[results["risk_type"] == risk_type].iterrows():
        sc_probs = row["raw_probs"]
        p_drop   = float(np.mean(sc_probs < (base_mean - threshold)))
        p_rise   = float(np.mean(sc_probs > (base_mean + threshold)))
        rows.append({
            "Scenario":                    row["scenario"],
            f"P(risk drops ≥{threshold:.0%})": p_drop,
            f"P(risk rises ≥{threshold:.0%})": p_rise,
            "Mean Risk":                   row["mean_risk"],
            "Δ vs Baseline":               row["mean_risk"] - base_mean,
        })
    return pd.DataFrame(rows)


for rt in ["cardiac", "diabetes"]:
    print(f"\n{'='*70}")
    print(f"  {'🫀 Cardiac' if rt == 'cardiac' else '🩸 Diabetes'} Risk — Probability Statements (threshold = 5%)")
    print(f"{'='*70}")
    df_stmt = compute_prob_statements(results, rt, threshold=0.05)
    print(df_stmt.to_string(index=False, float_format="{:.1%}".format))

In [ ]:
# ── 11. Plot E: Heatmap — Feature Sensitivity (Tornado Analysis) ──────────────
#
# For each risk type, shows how much each feature drives risk when perturbed ±1 SD.

def feature_sensitivity(
    baseline:  pd.DataFrame,
    model,
    features:  List[str],
    n_boot:    int = 3_000,
    seed:      int = 77,
) -> pd.Series:
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(baseline), size=n_boot, replace=False)
    sub = baseline[features].iloc[idx].values.astype(float)

    base_risk = np.mean(_safe_predict_proba(model, sub))
    deltas = {}

    for i, feat in enumerate(features):
        sd   = NHANES_PRIORS.get(feat, (0, 1, 0, 1))[1]
        hi   = sub.copy(); hi[:, i] += sd
        lo   = sub.copy(); lo[:, i] -= sd
        delta = (
            np.mean(_safe_predict_proba(model, hi)) -
            np.mean(_safe_predict_proba(model, lo))
        ) / (2 * sd + 1e-9)
        deltas[feat] = delta

    return pd.Series(deltas).sort_values(ascending=False)


fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Feature Sensitivity — Tornado Analysis (±1 SD Perturbation)",
             fontsize=13, fontweight="bold")

for ax, (model, features, title, cmap) in zip(axes, [
    (cardiac_model,  CARDIAC_FEATURES,  "🫀 Cardiac",  "RdPu"),
    (diabetes_model, DIABETES_FEATURES, "🩸 Diabetes", "BuPu"),
]):
    sens      = feature_sensitivity(baseline_cohort, model, features)
    colors_fn = plt.cm.get_cmap(cmap)
    vals      = sens.values
    norm_vals = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    bar_colors = [colors_fn(v * 0.7 + 0.2) for v in norm_vals]

    ax.barh(
        sens.index[::-1], sens.values[::-1],
        color=bar_colors[::-1], alpha=0.85,
        edgecolor="white", linewidth=0.4
    )
    ax.axvline(0, color="white", linewidth=0.8, alpha=0.5)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("ΔRisk per unit change (normalised)", fontsize=9)
    ax.grid(True, axis="x", alpha=0.2)

fig.tight_layout()
plt.show()

In [ ]:
# ── 12. Plot F: Violin Plot — Full Risk Distribution per Scenario ─────────────

def plot_violin(results: pd.DataFrame, risk_type: str):
    df     = results[results["risk_type"] == risk_type].reset_index(drop=True)
    labels = df["scenario"].tolist()
    data   = [row["raw_probs"] * 100 for _, row in df.iterrows()]
    colors = df["color"].tolist()

    fig, ax = plt.subplots(figsize=(15, 6))
    title_str = "🫀 Cardiac" if risk_type == "cardiac" else "🩸 Diabetes"
    ax.set_title(
        f"{title_str} Risk — Probability Distribution per Scenario (Violin)",
        fontsize=13, fontweight="bold"
    )

    parts = ax.violinplot(
        data, positions=range(len(data)),
        showmeans=True, showmedians=True,
        widths=0.7,
    )

    for i, (body, color) in enumerate(zip(parts["bodies"], colors)):
        body.set_facecolor(color)
        body.set_alpha(0.55)

    for key in ["cmeans", "cmedians", "cbars", "cmins", "cmaxes"]:
        if key in parts:
            parts[key].set_color("white")
            parts[key].set_linewidth(1.2)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
    ax.set_ylabel("Predicted Risk (%)", fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
    ax.grid(True, axis="y", alpha=0.2)

    fig.tight_layout()
    plt.show()


plot_violin(results, "cardiac")
plot_violin(results, "diabetes")

---
## 📝 Section 7 — Clinical Summary & Interpretation

In [ ]:
# ── 13. Auto-generated Clinical Narrative ─────────────────────────────────────

def generate_summary(results: pd.DataFrame) -> str:
    lines = ["\n" + "=" * 72]
    lines.append("  📋  TSUKUMO — WHAT-IF SCENARIO CLINICAL SUMMARY")
    lines.append("=" * 72)

    baseline_c = results[
        (results["scenario"] == "Current Trajectory") &
        (results["risk_type"] == "cardiac")
    ].iloc[0]
    baseline_d = results[
        (results["scenario"] == "Current Trajectory") &
        (results["risk_type"] == "diabetes")
    ].iloc[0]

    lines.append(f"""
BASELINE PATIENT
  Age: {PATIENT.age:.0f}  |  BMI: {PATIENT.bmi:.1f}  |  SBP: {PATIENT.sysBP:.0f} mmHg
  Glucose: {PATIENT.glucose:.0f} mg/dL  |  TotChol: {PATIENT.totChol:.0f} mg/dL

BASELINE RISK (Monte Carlo, N={N_SIMULATIONS:,})
  Cardiac Risk  : {baseline_c['mean_risk']:.1%}  (90% CI: {baseline_c['p05']:.1%} – {baseline_c['p95']:.1%})
  Diabetes Risk : {baseline_d['mean_risk']:.1%}  (90% CI: {baseline_d['p05']:.1%} – {baseline_d['p95']:.1%})
""")

    lines.append("SCENARIO RESULTS")
    lines.append("-" * 72)

    for sc in SCENARIOS[1:]:
        c_row = results[
            (results["scenario"] == sc.name) &
            (results["risk_type"] == "cardiac")
        ].iloc[0]
        d_row = results[
            (results["scenario"] == sc.name) &
            (results["risk_type"] == "diabetes")
        ].iloc[0]

        c_delta = c_row["mean_risk"] - baseline_c["mean_risk"]
        d_delta = d_row["mean_risk"] - baseline_d["mean_risk"]

        c_arrow = "⬇️ " if c_delta < 0 else "⬆️ "
        d_arrow = "⬇️ " if d_delta < 0 else "⬆️ "

        lines.append(f"""
  📌 {sc.name}
     {sc.description}
     Cardiac  : {c_row['mean_risk']:.1%}  {c_arrow}({c_delta:+.1%} vs baseline)  90% CI [{c_row['p05']:.1%}, {c_row['p95']:.1%}]
     Diabetes : {d_row['mean_risk']:.1%}  {d_arrow}({d_delta:+.1%} vs baseline)  90% CI [{d_row['p05']:.1%}, {d_row['p95']:.1%}]""")

    lines.append("\n" + "=" * 72)
    lines.append("  ⚠️  DISCLAIMER: This is a statistical simulation tool, not medical advice.")
    lines.append("       Consult a licensed physician before making clinical decisions.")
    lines.append("=" * 72)
    return "\n".join(lines)


print(generate_summary(results))

---
## 🔧 Section 8 — Interactive Custom Scenario Builder

Use the sliders below to define your own what-if scenario in real time.

In [ ]:
# ── 14. Interactive Widget ─────────────────────────────────────────────────────
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    style = {"description_width": "200px"}
    layout = widgets.Layout(width="500px")

    w_sysbp  = widgets.IntSlider(value=int(PATIENT.sysBP),  min=80, max=220, step=1,  description="Systolic BP (mmHg)",        style=style, layout=layout)
    w_bmi    = widgets.FloatSlider(value=PATIENT.bmi,       min=15, max=60,  step=0.5,description="BMI",                        style=style, layout=layout)
    w_glu    = widgets.IntSlider(value=int(PATIENT.glucose),min=60, max=400, step=1,  description="Glucose (mg/dL)",           style=style, layout=layout)
    w_sleep  = widgets.FloatSlider(value=PATIENT.sleep_hours,min=3, max=12,  step=0.5,description="Sleep Hours",               style=style, layout=layout)
    w_met    = widgets.FloatSlider(value=PATIENT.physical_activity_met, min=1, max=15, step=0.5, description="Physical Activity (MET)", style=style, layout=layout)
    w_sodium = widgets.IntSlider(value=int(PATIENT.sodium_mg_per_day), min=500, max=8000, step=100, description="Sodium (mg/day)",      style=style, layout=layout)
    w_chol   = widgets.IntSlider(value=int(PATIENT.totChol),min=100,max=400, step=5,  description="Total Cholesterol (mg/dL)", style=style, layout=layout)
    btn      = widgets.Button(description="▶ Run My Scenario", button_style="info", icon="play",
                               layout=widgets.Layout(width="200px", margin="12px 0px"))
    out      = widgets.Output()

    def on_run(_):
        with out:
            clear_output(wait=True)
            custom_patient = PatientProfile(
                sysBP               = w_sysbp.value,
                bmi                 = w_bmi.value,
                glucose             = w_glu.value,
                sleep_hours         = w_sleep.value,
                physical_activity_met = w_met.value,
                sodium_mg_per_day   = w_sodium.value,
                totChol             = w_chol.value,
                # Keep other fields from global PATIENT
                age                 = PATIENT.age,
                diaBP               = PATIENT.diaBP,
                heartRate           = PATIENT.heartRate,
                HDLChol             = PATIENT.HDLChol,
                cigarettes_per_day  = PATIENT.cigarettes_per_day,
                alcohol_drinks_per_week = PATIENT.alcohol_drinks_per_week,
                prevalentHyp        = PATIENT.prevalentHyp,
                diabetes            = PATIENT.diabetes,
                currentSmoker       = PATIENT.currentSmoker,
            )
            cohort = generate_cohort(custom_patient, n=5_000)

            # Single-point predictions
            Xc = cohort[CARDIAC_FEATURES].values.astype(float)
            Xd = cohort[DIABETES_FEATURES].values.astype(float)
            cp = _safe_predict_proba(cardiac_model, Xc)
            dp = _safe_predict_proba(diabetes_model, Xd)

            print(f"""
╔══════════════════════════════════════════════════╗
║     🧬  YOUR CUSTOM WHAT-IF SCENARIO RESULT      ║
╠══════════════════════════════════════════════════╣
║  Inputs:  SBP={w_sysbp.value} | BMI={w_bmi.value:.1f} | Glucose={w_glu.value}  ║
║           Sleep={w_sleep.value}h | MET={w_met.value:.1f} | Na={w_sodium.value}mg  ║
╠══════════════════════════════════════════════════╣
║  🫀 Cardiac Risk   : {np.mean(cp):.1%}  (90% CI: {np.percentile(cp,5):.1%}–{np.percentile(cp,95):.1%})  ║
║  🩸 Diabetes Risk  : {np.mean(dp):.1%}  (90% CI: {np.percentile(dp,5):.1%}–{np.percentile(dp,95):.1%})  ║
╚══════════════════════════════════════════════════╝
            """)

            # Mini bar chart
            fig, ax = plt.subplots(figsize=(7, 3))
            risks   = [np.mean(cp), np.mean(dp)]
            labels  = ["🫀 Cardiac", "🩸 Diabetes"]
            colors  = ["#e74c3c", "#3498db"]
            bars    = ax.bar(labels, [r * 100 for r in risks], color=colors, alpha=0.8, width=0.4)
            for bar, r in zip(bars, risks):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                        f"{r:.1%}", ha="center", fontsize=11, fontweight="bold", color="white")
            ax.set_ylabel("Predicted Risk (%)")
            ax.set_title("Your Custom Scenario — Predicted Risk", fontsize=11)
            ax.set_ylim(0, max([r * 100 for r in risks]) * 1.4 or 10)
            ax.grid(True, axis="y", alpha=0.2)
            fig.tight_layout()
            plt.show()

    btn.on_click(on_run)

    display(
        widgets.VBox([
            widgets.HTML("<h3 style='color:#7b7bff'>🎛️  Custom What-If Scenario Builder</h3>"),
            widgets.HTML("<p style='color:#aaa'>Adjust the sliders, then click ▶ Run My Scenario</p>"),
            w_sysbp, w_bmi, w_glu, w_sleep, w_met, w_sodium, w_chol,
            btn,
            out,
        ])
    )

except ImportError:
    print("⚠️  ipywidgets not available — run:  pip install ipywidgets")
    print("   Then re-execute this cell for the interactive slider UI.")

---
## 💾 Section 9 — Export Results

In [ ]:
# ── 15. Export to CSV ─────────────────────────────────────────────────────────
export_cols = ["scenario", "risk_type", "mean_risk", "p05", "p25", "p50", "p75", "p95"]
export_df   = results[export_cols].copy()
export_df.to_csv("whatif_results.csv", index=False)
print("✅  Exported to whatif_results.csv")

# ── Clinical summary text ─────────────────────────────────────────────────────
with open("whatif_summary.txt", "w", encoding="utf-8") as f:
    f.write(generate_summary(results))
print("✅  Clinical summary written to whatif_summary.txt")

print("\nAll outputs saved in:", os.path.abspath("."))